In [16]:
#importing our libraries
from bs4 import BeautifulSoup
import pandas as pd
import requests

In [2]:
BBC_URL = "https://www.bbc.com/news/topics/cwlw3xz047jt"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

In [13]:
Articles = []

for page in range(1,43):
    print(f"Fetching Page {page} ...")
    url = BBC_URL if page == 1 else f"{BBC_URL}?page={page}"
    
    htmlDoc = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(htmlDoc.content, "html.parser")

    articles = soup.find_all("a", {"data-testid": "internal-link"}) or soup.find_all("a", href=True)

    for article in articles:
        href = article.get("href")
        if not href or not any(path in href for path in ["/news/articles/", "/news/world-", "/news/blogs"]):
            continue

        title_tag = article.find("h2") or article.find("span")
        title = title_tag.text.strip() if title_tag else "No Title"
        article_link = href if href.startswith("http") else f"https://www.bbc.com{href}"

        description_tag = article.find("p")
        description = description_tag.text.strip() if description_tag else "No Description"

        # Article Details page
        articleDetailPage = requests.get(article_link, headers=HEADERS)
        article_soup = BeautifulSoup(articleDetailPage.content, "html.parser")

        # Banner Image 
        og_tag = article_soup.find("meta", property="og:image") or article_soup.find("meta", attrs={"name": "og:image"})
        banner = og_tag.get("content") if og_tag else "No Banner Image"

        # Author and Contributor
        author_tag = article_soup.select_one('span[class*="AuthorNameStyled"]')
        contrib_tag = article_soup.select_one('span[data-testid*="role-location"]')

        author = author_tag.text.strip() if author_tag else "Unknown"
        contributor = contrib_tag.text.strip() if contrib_tag else "Unknown"

        # Main Content Block Extraction 
        body = article_soup.find("article") or article_soup
        text_blocks = body.select('div[data-component="text-block"] p')
        content_segments = [p.text.strip() for p in text_blocks if p.text.strip()]

        Articles.append({
            "Title": title,
            "Description": description,
            "Link": article_link,
            "Banner Image": banner,
            "Author": author,
            "Contributor": contributor,
            "Content": "\n\n".join(content_segments) or "No content found",
        })

print("\n Done!")

Fetching Page 1 ...
Fetching Page 2 ...
Fetching Page 3 ...
Fetching Page 4 ...
Fetching Page 5 ...
Fetching Page 6 ...
Fetching Page 7 ...
Fetching Page 8 ...
Fetching Page 9 ...
Fetching Page 10 ...
Fetching Page 11 ...
Fetching Page 12 ...
Fetching Page 13 ...
Fetching Page 14 ...
Fetching Page 15 ...
Fetching Page 16 ...
Fetching Page 17 ...
Fetching Page 18 ...
Fetching Page 19 ...
Fetching Page 20 ...
Fetching Page 21 ...
Fetching Page 22 ...
Fetching Page 23 ...
Fetching Page 24 ...
Fetching Page 25 ...
Fetching Page 26 ...
Fetching Page 27 ...
Fetching Page 28 ...
Fetching Page 29 ...
Fetching Page 30 ...
Fetching Page 31 ...
Fetching Page 32 ...
Fetching Page 33 ...
Fetching Page 34 ...
Fetching Page 35 ...
Fetching Page 36 ...
Fetching Page 37 ...
Fetching Page 38 ...
Fetching Page 39 ...
Fetching Page 40 ...
Fetching Page 41 ...
Fetching Page 42 ...

 Done!


In [14]:
df = pd.DataFrame(Articles)
df

,Title,Description,Link,Banner Image,Author,Contributor,Content
0,Voting suspended in parts of Ethiopia over sec...,Prime Minister Abiy Ahmed's party is expected ...,https://www.bbc.com/news/articles/cn0pngz2rego,https://ichef.bbci.co.uk/news/1024/branded_new...,Kaleb Moges,BBC Africa,Voting in Monday's general election was suspen...
1,I moved from Ethiopia to Shetland - and I've b...,Netsanet Sori is bringing a flavour of the far...,https://www.bbc.com/news/articles/cvgz46znz0vo,https://ichef.bbci.co.uk/news/1024/branded_new...,Iona Nicol,BBC Scotland News,Shetland has plenty of honesty boxes - filled ...
2,Ethiopian woman's joy at rare quintuplets afte...,"The woman, 35, says she was praying for a baby...",https://www.bbc.com/news/articles/c893y34zlv0o,https://ichef.bbci.co.uk/news/1024/branded_new...,Michael Teferi,BBC Africa,A woman has given birth to rare quintuplets in...
3,Millions listen to Ethiopian star's song takin...,The country's biggest musician Teddy Afro lame...,https://www.bbc.com/news/articles/cqj81g5dxkyo,https://ichef.bbci.co.uk/news/1024/branded_new...,BBC Amharic,Unknown,In a country where criticising the authorities...
4,Ringleader of suspected human trafficking netw...,Yetbarek Dawit is alleged to have operated a m...,https://www.bbc.com/news/articles/cn08gx8k5ego,https://ichef.bbci.co.uk/news/1024/branded_new...,Kaleb Moges,BBC Africa,Police in Ethiopia have arrested a man accused...
...,...,...,...,...,...,...,...
342,Seismic change: How Covid-19 altered world eve...,The pandemic affected us all - and shaped glob...,https://www.bbc.com/news/world-55233822,https://ichef.bbci.co.uk/news/1024/branded_new...,Helier Cheung,BBC News,The year 2020 has been like no other. The coro...
343,Somalia's Christmas birthdays and lost memories,A former refugee considers the importance of m...,https://www.bbc.com/news/world-africa-55333537,https://ichef.bbci.co.uk/news/1024/branded_new...,Unknown,Unknown,In our series of letters from African journali...
344,Ethiopia military 'kills 40 after 100 massacred',Children were among more than 100 people kille...,https://www.bbc.com/news/world-africa-55428322,https://ichef.bbci.co.uk/news/1024/branded_new...,Unknown,Unknown,Ethiopia's military has killed more than 40 me...
345,Attackers kill dozens in western Ethiopia,More than 100 people were killed in fires or b...,https://www.bbc.com/news/world-africa-55428531,https://ichef.bbci.co.uk/news/1024/branded_new...,Unknown,Unknown,More than 100 people have been killed by unkno...


In [15]:
# exporting our dataframe into a csv file
df.to_csv("bbc_news.csv", index=False, encoding="utf-8-sig")